# 🦙 LlamaIndex Master Course - Level 3
## Documents & Nodes (Deep Dive)

Welcome to Level 3! In this notebook, we will explore the internal anatomy of `Document` and `Node` objects, understand how metadata flows through the pipeline, test various node parsers, and build a complete extraction pipeline.

In [ ]:
# Setup - Run this cell first!
!pip install -q llama-index-core llama-index-readers-file tiktoken

import os
from dotenv import load_dotenv

load_dotenv() # Load your OpenAI API key if you have one

print("Libraries loaded successfully!")

### 1. The Document Object Anatomy

Let's create a Document with rich metadata and inspect what the LLM and Embedding models will actually see.

In [ ]:
from llama_index.core import Document
from llama_index.core.schema import MetadataMode

# Create Document with all important fields
doc = Document(
    text="""TechNova Q4 2024 Executive Summary\n\nRevenue reached $7.175M, representing 86% YoY growth.\nEBITDA of $1.945M yielded a 27.1% operating margin.\nWe acquired 47 new enterprise customers including 5 Fortune 500.\nNPS improved from 61 to 68, reflecting strong product-market fit.\nThe Singapore office expansion added 12 engineers.""",
    doc_id="technova-q4-2024-exec-summary",
    metadata={
        "source"         : "q4_2024_board_deck.pdf",
        "doc_type"       : "executive_summary",
        "quarter"        : "Q4",
        "year"           : "2024",
        "department"     : "Executive",
        "company"        : "TechNova Inc.",
        "author"         : "CEO Sarah Chen",
        "classification" : "board_confidential",
    },
    # Exclusions
    excluded_embed_metadata_keys=["author", "classification"],
    excluded_llm_metadata_keys=["classification"],
    # Custom template
    metadata_template="[{key}]: {value}",
    metadata_separator="\n",
)

print("=== LLM Context (What LLM Sees) ===")
print(doc.get_content(metadata_mode=MetadataMode.LLM))
print("\n" + "="*40 + "\n")
print("=== Embedding Context (What Embedding Model Sees) ===")
print(doc.get_content(metadata_mode=MetadataMode.EMBED))

### 2. Node Parsing & Relationship Tracking

We will now parse this document into smaller Nodes and examine the automatically generated relationships (`SOURCE`, `PREVIOUS`, `NEXT`).

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import NodeRelationship

# Parse into Nodes using a small chunk size for demonstration
parser = SentenceSplitter(chunk_size=40, chunk_overlap=10)
nodes = parser.get_nodes_from_documents([doc])

print(f"Document split into {len(nodes)} nodes.\n")

for i, node in enumerate(nodes):
    print(f"--- NODE {i+1} ---")
    print(f"ID: {node.node_id[:15]}...")
    print(f"Text: {node.text.strip()[:60]}...")
    
    print("Relationships:")
    if NodeRelationship.SOURCE in node.relationships:
        print(f"  SOURCE   -> {node.relationships[NodeRelationship.SOURCE].node_id}")
    if NodeRelationship.PREVIOUS in node.relationships:
        print(f"  PREVIOUS -> {node.relationships[NodeRelationship.PREVIOUS].node_id[:15]}...")
    if NodeRelationship.NEXT in node.relationships:
        print(f"  NEXT     -> {node.relationships[NodeRelationship.NEXT].node_id[:15]}...")
    print()

### 3. Comparing Different Parsers

LlamaIndex provides multiple node parsers. Let's see how `SentenceSplitter`, `TokenTextSplitter`, and `HierarchicalNodeParser` differ.

In [ ]:
from llama_index.core.node_parser import TokenTextSplitter, HierarchicalNodeParser, get_leaf_nodes

# 1. SentenceSplitter (Respects sentence boundaries)
sentence_parser = SentenceSplitter(chunk_size=30, chunk_overlap=0)
nodes_sentence = sentence_parser.get_nodes_from_documents([doc])

# 2. TokenTextSplitter (Hard splits by tokens)
token_parser = TokenTextSplitter(chunk_size=30, chunk_overlap=0)
nodes_token = token_parser.get_nodes_from_documents([doc])

# 3. HierarchicalNodeParser (Multi-level parent-child)
hier_parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[100, 30])
nodes_hier = hier_parser.get_nodes_from_documents([doc])

print("SentenceSplitter chunks (Notice how it keeps sentences intact):")
for i, n in enumerate(nodes_sentence[:3]):
    print(f"  [{i+1}] {n.text.strip()}")
print("\n" + "="*60 + "\n")
print("TokenTextSplitter chunks (Notice hard cuts mid-sentence):")
for i, n in enumerate(nodes_token[:3]):
    print(f"  [{i+1}] {n.text.strip()}")
print("\n" + "="*60 + "\n")
print(f"Hierarchical Parser generated {len(nodes_hier)} total nodes.")
leaf_nodes = get_leaf_nodes(nodes_hier)
print(f"Of those, {len(leaf_nodes)} are leaf (child) nodes and the rest are parents.")

### 4. Custom Node Transformations

You can modify Nodes after parsing but before indexing. Let's write a simple rule-based transformation to tag financial data without using an LLM.

In [ ]:
import re
from llama_index.core.schema import TextNode

def detect_financial_metrics(nodes: list[TextNode]) -> list[TextNode]:
    """Custom node transformation to tag financial text."""
    for node in nodes:
        text_lower = node.text.lower()
        # Check for dollar amounts or financial terms
        if re.search(r'\$[\d,.]+[MmBb]|\brevenue\b|\bebitda\b', text_lower):
            node.metadata["contains_financials"] = "True"
        else:
            node.metadata["contains_financials"] = "False"
            
        # Don't embed this computed flag, just use it for filtering/LLM
        if "contains_financials" not in node.excluded_embed_metadata_keys:
            node.excluded_embed_metadata_keys.append("contains_financials")
            
    return nodes

# Apply it to our SentenceSplitter nodes
transformed_nodes = detect_financial_metrics(nodes_sentence)

for i, n in enumerate(transformed_nodes):
    print(f"Node {i+1}: {n.text.strip()[:40]}...")
    print(f"  contains_financials: {n.metadata.get('contains_financials')}")
    print()

### 🎉 Level 3 Complete!
You now know how to deeply manipulate Documents and Nodes. This prepares you for Level 4 where we will start building robust Vector Stores and Indices.